In [0]:
from pyspark.sql import functions as F

# Read the yellow taxi table and print basic info: row count, column count, schema, and summary stats
df = spark.read.table("group3_gp.testing.yellow")
print(f"Row count : {df.count():,}")
print(f"Columns   : {len(df.columns)}")
df.printSchema()
df.describe().show()

In [0]:
%sql
-- ════════════════════════════════════════════════
-- YELLOW — Bad data checks
-- Counts rows failing each quality rule in one pass
-- ════════════════════════════════════════════════

-- Row count
SELECT 'Total rows' AS check_name, COUNT(*) AS result
FROM group3_gp.testing.yellow

UNION ALL

-- Bad fares
-- EXCEPTION: payment_type 3 = No charge, 6 = Voided — zero fare is CORRECT
SELECT 'Bad fares (excl. no-charge & voided)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.yellow
WHERE CAST(fare_amount AS DOUBLE) <= 0
  AND payment_type NOT IN ('3', '6')

UNION ALL

-- Legitimate zero fares
SELECT 'Legitimate zero fares (no-charge/voided)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.yellow
WHERE CAST(fare_amount AS DOUBLE) <= 0
  AND payment_type IN ('3', '6')

UNION ALL

-- Bad distances
SELECT 'Bad distances (zero or negative)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.yellow
WHERE CAST(trip_distance AS DOUBLE) <= 0

UNION ALL

-- Bad timestamps — coalesce handles both old and new era column names
SELECT 'Bad timestamps (dropoff <= pickup)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.yellow
WHERE 
COALESCE(TO_TIMESTAMP(tpep_dropoff_datetime), TO_TIMESTAMP(dropoff_datetime)) IS NULL
OR COALESCE(TO_TIMESTAMP(tpep_pickup_datetime), TO_TIMESTAMP(pickup_datetime)) IS NULL
OR COALESCE(TO_TIMESTAMP(tpep_dropoff_datetime), TO_TIMESTAMP(dropoff_datetime))
   <= COALESCE(TO_TIMESTAMP(tpep_pickup_datetime), TO_TIMESTAMP(pickup_datetime))

UNION ALL

-- Invalid RatecodeID — 99 = Null/unknown per data dictionary
SELECT 'Invalid ratecode (99 = unknown)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.yellow
WHERE COALESCE(TRY_CAST(ratecodeid AS INT), TRY_CAST(rate_code AS INT)) = 99

UNION ALL

-- Stored offline — meter lost connection, may affect data quality
SELECT 'Stored offline trips (store_and_fwd_flag = Y)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.yellow
WHERE store_and_fwd_flag = 'Y'

UNION ALL

-- Invalid passenger counts
SELECT 'Invalid passenger count (< 1 or > 8)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.yellow
WHERE TRY_CAST(passenger_count AS INT) NOT BETWEEN 1 AND 8

UNION ALL

-- Duplicates — proxy using key columns, not full row DISTINCT
SELECT 'Duplicate rows' AS check_name,
       COUNT(*) - COUNT(DISTINCT
           CONCAT_WS('|',
               COALESCE(tpep_pickup_datetime, pickup_datetime),
               COALESCE(vendorid, vendor_id),
               fare_amount,
               trip_distance
           )
       ) AS result
FROM group3_gp.testing.yellow

In [0]:
%sql
-- Yellow — Payment type breakdown

SELECT
    payment_type,
    CASE payment_type
        WHEN '0' THEN 'Flex fare'
        WHEN '1' THEN 'Credit card'
        WHEN '2' THEN 'Cash'
        WHEN '3' THEN 'No charge'
        WHEN '4' THEN 'Dispute'
        WHEN '5' THEN 'Unknown'
        WHEN '6' THEN 'Voided trip'
        WHEN 'CRD' THEN 'Credit card'
        WHEN 'CSH' THEN 'Cash'
        WHEN 'UNK' THEN 'Unknown'
        WHEN 'NOC' THEN 'No charge'
        WHEN 'DIS' THEN 'Dispute'
        WHEN 'VOID' THEN 'Voided trip'
        ELSE 'Unrecognised'
    END AS payment_description,
    COUNT(*) AS trip_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM group3_gp.testing.yellow
GROUP BY payment_type
ORDER BY trip_count DESC

In [0]:
%sql
-- Yellow — Rate code breakdown
-- Coalesces old (rate_code) and new (ratecodeid) column names
SELECT
    COALESCE(ratecodeid, rate_code) AS ratecode,

    CASE 
        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) = 1 THEN 'Standard rate'
        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) = 2 THEN 'JFK'
        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) = 3 THEN 'Newark'
        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) = 4 THEN 'Nassau or Westchester'
        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) = 5 THEN 'Negotiated fare'
        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) = 6 THEN 'Group ride'
        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) = 99 THEN 'Unknown / invalid'

        WHEN TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) IS NULL 
             AND COALESCE(ratecodeid, rate_code) IS NOT NULL THEN 'String'

        ELSE 'Unrecognised'
    END AS rate_description,

    COUNT(*) AS trip_count

FROM group3_gp.testing.yellow

GROUP BY ratecode

ORDER BY ratecode ASC

In [0]:
%sql
-- Yellow — Vendor breakdown
-- Coalesces old (vendor_id) and new (vendorid) column names

SELECT
    COALESCE(vendorid, vendor_id) AS vendor_id,
    CASE try_cast(COALESCE(vendorid, vendor_id) AS INT)
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'Curb Mobility'
        WHEN 6 THEN 'Myle Technologies'
        WHEN 7 THEN 'Helix'
        ELSE 'Unknown'
    END AS vendor_name,
    COUNT(*) AS trip_count
FROM group3_gp.testing.yellow
GROUP BY COALESCE(vendorid, vendor_id)
ORDER BY trip_count DESC

In [0]:
# Deep dive into bad fares: zero vs negative, distance presence, payment type, year, and vendor breakdown
df = spark.read.table("group3_gp.testing.yellow")

# Is it mostly zero fares or actually negative?
print("Zero fare:    ", df.filter(F.col("fare_amount").cast("double") == 0).count())
print("Negative fare:", df.filter(F.col("fare_amount").cast("double") <  0).count())

# For zero-fare trips — do they have a distance? 
# If distance > 0 but fare = 0, likely a cash/meter error
display(
    df.filter(F.col("fare_amount").cast("double") == 0)
      .withColumn("trip_distance", F.col("trip_distance").cast("double"))
      .groupBy(
          F.when(F.col("trip_distance") > 0, "has distance")
           .otherwise("no distance").alias("distance_status")
      ).count()
)

# For zero-fare trips — what payment type are they?
# payment_type 2 = cash, 1 = card
# Cash trips are far more likely to have $0 fare logged
display(
    df.filter(F.col("fare_amount").cast("double") == 0)
      .groupBy("payment_type")
      .count()
      .orderBy(F.col("count").desc())
)

# Are bad fares concentrated in specific years?
# Schema transition years (2016-2017) often have spikes
display(
    df.filter(F.col("fare_amount").cast("double") <= 0)
      .groupBy("Year")
      .count()
      .orderBy("Year")
)
# Are bad fares concentrated in specific vendors?
# A single bad vendor system causes spikes
display(
    df.filter(F.col("fare_amount").cast("double") <= 0)
      .groupBy("vendorid")
      .count()
      .orderBy(F.col("count").desc())
)

## Yellow Taxi Data Quality — Summary

### Dataset Overview

| Property | Value |
| --- | --- |
| Table | `group3_gp.testing.yellow` |
| Rows | 890,945,235 |
| Columns | 31 |
| Time span | 2011–2023 |
| Schema note | All numeric columns stored as `STRING` — requires explicit casting |

---

### Data Quality Checks

| Check | Count | % of Total |
| --- | ---: | ---: |
| Bad fares (excl. no-charge & voided) | 754,287 | 0.08% |
| Legitimate zero fares (no-charge / voided) | 249,905 | 0.03% |
| Bad distances (zero or negative) | 6,845,654 | 0.77% |
| Bad timestamps (dropoff ≤ pickup) | 1,772,754 | 0.20% |
| Invalid ratecode (99) | 18,553 | <0.01% |
| Stored offline (store_and_fwd_flag = Y) | 8,434,814 | 0.95% |
| Invalid passenger count (<1 or >8) | 2,697,961 | 0.30% |
| Duplicate rows (proxy) | 9,725,548 | 1.09% |

> **Ratecode check gap:** The current check uses `TRY_CAST(... AS INT)`, which silently returns NULL for decimal-formatted strings like `"99.0"`. This catches only 18,553 rows (`"99"`), missing an additional 347,274 rows stored as `"99.0"`. True total: **365,827**. Fix: use `TRY_CAST(... AS DOUBLE) = 99`.

---

### Payment Types

Legacy string codes (`CRD`, `CSH`, `NOC`, `DIS`, `UNK`) from the pre-2017 era coexist with numeric codes. Combined:

| Payment method | Trips | Share |
| --- | ---: | ---: |
| Credit card (1 + CRD) | 528,482,486 | 59.32% |
| Cash (2 + CSH) | 353,704,214 | 39.70% |
| Flex fare (0) | 2,677,659 | 0.30% |
| No charge (3 + NOC) | 2,672,358 | 0.30% |
| Dispute (4 + DIS) | 1,497,910 | 0.17% |
| Unknown (UNK + 5) / NULL | 1,910,608 | 0.21% |

---

### Rate Codes

Values stored inconsistently as both `"1"` and `"1.0"` across eras. Combined:

| Rate code | Trips |
| --- | ---: |
| Standard rate (1) | 863,116,587 |
| JFK (2) | 18,507,226 |
| Negotiated fare (5) | 2,936,163 |
| Newark (3) | 1,553,436 |
| Nassau / Westchester (4) | 806,149 |
| Group ride (6) | 10,278 |
| Unknown / invalid (99) | 365,827 |
| NULL | 3,585,579 |
| Other unrecognised codes | \~63,990 |

---

### Vendors

Two legacy string codes (`VTS`, `CMT`) represent \~53% of trips from the pre-2017 era.

| Vendor | Trips | Share |
| --- | ---: | ---: |
| Curb Mobility (2) | 240,268,506 | 26.97% |
| VTS (legacy) | 237,408,719 | 26.65% |
| CMT (legacy) | 236,221,325 | 26.51% |
| Creative Mobile Technologies (1) | 175,987,369 | 19.75% |
| NULL / Other | 1,059,316 | 0.12% |

---

### Bad Fares — Deep Dive

**Negative vs zero:** Negative fares (867,418) outnumber zero fares (137,478) by roughly 6:1.

**Zero-fare trips with distance:** 57% of zero-fare trips (77,812 of 137,478) still have a recorded trip distance — likely meter or logging errors rather than genuinely free rides.

**Zero-fare payment types:** Cash dominates (64,325), followed by no-charge (31,245) and credit card (27,746).

**Year concentration:** Bad fares spike in 2022–2023 (664,677 combined, 66% of all bad fares). Smaller clusters appear around 2015–2016 and 2020.

| Year | Bad fare count |
| --- | ---: |
| 2023 | 394,503 |
| 2022 | 270,174 |
| 2020 | 103,827 |
| 2016 | 93,133 |
| 2015 | 92,912 |
| Others | \~50,347 |

**Vendor concentration:** Curb Mobility (vendor 2) accounts for **84%** of all bad fares (842,249 of \~1,004,896). This is disproportionate given it handles only 27% of total trips.